In [1]:
import os
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import warnings

# 关闭 Python warning
warnings.filterwarnings("ignore")

# 关闭 RDKit warning / error log
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


# =========================================================
# 1. 全局配置
# =========================================================
data_file = "../invertebrates_LC50_unique.xlsx"
smiles_col = "SMILES_Canonical_RDKit"
label_col = "mgperL"

pred_dir = "../k_folds_model/xgb_predictions"
n_splits = 10

# Morgan 指纹参数
radius = 2
n_bits = 2048

# 数值稳定项
ridge_alpha = 1e-6

# 阈值扫描范围（可按需要调整）
# leverage 通常不在 0-1 内，所以这里后面会根据全部测试样本的 leverage 自动生成阈值
GRID_SIZE = 200

# 预测文件名模板
pred_file_template = "fold_{fold}_y_pred.npy"


# =========================================================
# 2. 读取数据
# =========================================================
data = pd.read_excel(data_file)
data = data.dropna(subset=[smiles_col, label_col]).copy()

smiles_data = data[smiles_col].tolist()
y_all = np.log1p(data[label_col].values.astype(float))   # 与训练时一致
groups = smiles_data


# =========================================================
# 3. Morgan fingerprint 工具函数
# =========================================================
def smiles_to_morgan_array(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    from rdkit import DataStructs
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def batch_smiles_to_fp_array(smiles_list, radius=2, n_bits=2048):
    X = []
    valid_idx = []
    for i, smi in enumerate(smiles_list):
        arr = smiles_to_morgan_array(smi, radius=radius, n_bits=n_bits)
        if arr is not None:
            X.append(arr)
            valid_idx.append(i)
    if len(X) == 0:
        return np.empty((0, n_bits), dtype=np.float32), np.array([], dtype=int)
    return np.vstack(X).astype(np.float32), np.array(valid_idx, dtype=int)


# =========================================================
# 4. Leverage 计算函数
# =========================================================
def compute_leverage_scores(X_train, X_test, ridge_alpha=1e-6):
    """
    计算测试集样本相对于训练集的 leverage 值
    h_i = x_i^T (X^T X)^(-1) x_i

    参数
    ----
    X_train: shape [n_train, p]
    X_test : shape [n_test, p]

    返回
    ----
    leverage_scores: shape [n_test]
    """
    # 转 float64 提高数值稳定性
    X_train = X_train.astype(np.float64)
    X_test = X_test.astype(np.float64)

    p = X_train.shape[1]

    XtX = X_train.T @ X_train
    XtX_reg = XtX + ridge_alpha * np.eye(p, dtype=np.float64)

    XtX_inv = np.linalg.pinv(XtX_reg)

    # 对每个测试样本计算 leverage
    # h_i = x_i^T XtX_inv x_i
    leverage_scores = np.einsum('ij,jk,ik->i', X_test, XtX_inv, X_test)

    return leverage_scores


# =========================================================
# 5. 重建与你训练时一致的 10 折划分
# =========================================================
gkf = GroupKFold(n_splits=n_splits)
fold_splits = list(gkf.split(smiles_data, y_all, groups))


# =========================================================
# 6. 检查预测文件
# =========================================================
for fold_id in range(1, n_splits + 1):
    pred_path = os.path.join(pred_dir, pred_file_template.format(fold=fold_id))
    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"未找到预测文件: {pred_path}")


# =========================================================
# 7. 收集所有折的 leverage AD 分数
# =========================================================
fold_cache = []
all_scores = []

for fold_id, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    train_smiles = [smiles_data[i] for i in train_idx]
    val_smiles   = [smiles_data[i] for i in val_idx]

    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]

    pred_path = os.path.join(pred_dir, pred_file_template.format(fold=fold_id))
    y_pred = np.load(pred_path)

    # 转为 Morgan fingerprint 数组
    X_train, train_valid_idx = batch_smiles_to_fp_array(train_smiles, radius=radius, n_bits=n_bits)
    X_val, val_valid_idx     = batch_smiles_to_fp_array(val_smiles, radius=radius, n_bits=n_bits)

    # 若存在无效 SMILES，同步过滤
    if len(train_valid_idx) != len(train_smiles):
        y_train = y_train[train_valid_idx]
        train_smiles = [train_smiles[i] for i in train_valid_idx]

    if len(val_valid_idx) != len(val_smiles):
        y_val = y_val[val_valid_idx]
        y_pred = y_pred[val_valid_idx]
        val_smiles = [val_smiles[i] for i in val_valid_idx]

    # 计算 leverage 分数
    leverage_scores = compute_leverage_scores(X_train, X_val, ridge_alpha=ridge_alpha)

    all_scores.append(leverage_scores)

    fold_cache.append({
        "fold_id": fold_id,
        "train_smiles": train_smiles,
        "val_smiles": val_smiles,
        "y_train": y_train,
        "y_val": y_val,
        "y_pred": y_pred,
        "ad_score": leverage_scores
    })

all_scores = np.concatenate(all_scores)


# =========================================================
# 8. 阈值扫描：输出不同阈值下的十折平均 R²_in
#    并增加覆盖率指标：Coverage（保留两位小数）
# =========================================================
# Leverage 越小越“在域内”，所以阈值从小到大全扫
score_cuts = np.linspace(all_scores.min(), all_scores.max(), 21)

results = []

for th in score_cuts:
    fold_r2_list = []
    retained_list = []
    total_list = []

    for cache in fold_cache:
        score = cache["ad_score"]
        y_val = cache["y_val"]
        y_pred = cache["y_pred"]

        # Leverage-based AD: leverage <= threshold 判为 in-AD
        in_mask = score <= th

        retained_n = int(in_mask.sum())
        total_n = int(len(in_mask))

        retained_list.append(retained_n)
        total_list.append(total_n)

        if retained_n > 1:
            r2_in = r2_score(y_val[in_mask], y_pred[in_mask])
        else:
            r2_in = np.nan

        fold_r2_list.append(r2_in)

    mean_r2 = np.nanmean(fold_r2_list)
    std_r2 = np.nanstd(fold_r2_list)

    mean_retained = np.mean(retained_list)
    mean_total = np.mean(total_list)
    coverage = mean_retained / mean_total if mean_total > 0 else np.nan

    results.append({
        "threshold": round(float(th), 6),
        "mean_R2_in_10fold": round(float(mean_r2), 4) if not np.isnan(mean_r2) else np.nan,
        "std_R2_in_10fold": round(float(std_r2), 4) if not np.isnan(std_r2) else np.nan,
        "Coverage": round(float(coverage), 2) if not np.isnan(coverage) else np.nan
    })

results_df = pd.DataFrame(results)

print(results_df)
display(results_df)

best_row = results_df.loc[results_df["mean_R2_in_10fold"].idxmax()]

print("=" * 80)
print("Leverage-based AD 最佳阈值结果：")
print(best_row)
print("=" * 80)


libgomp: Invalid value for environment variable OMP_NUM_THREADS


       threshold  mean_R2_in_10fold  std_R2_in_10fold  Coverage
0   6.355100e-02            -0.0970            0.0000      0.00
1   1.089415e+06             0.6686            0.0871      0.56
2   2.178829e+06             0.6781            0.0686      0.74
3   3.268244e+06             0.6589            0.0600      0.86
4   4.357658e+06             0.6501            0.0618      0.92
5   5.447072e+06             0.6426            0.0550      0.95
6   6.536487e+06             0.6377            0.0468      0.97
7   7.625901e+06             0.6362            0.0478      0.98
8   8.715316e+06             0.6354            0.0478      0.99
9   9.804730e+06             0.6349            0.0481      0.99
10  1.089414e+07             0.6355            0.0479      1.00
11  1.198356e+07             0.6355            0.0480      1.00
12  1.307297e+07             0.6355            0.0480      1.00
13  1.416239e+07             0.6354            0.0480      1.00
14  1.525180e+07             0.6355     

,threshold,mean_R2_in_10fold,std_R2_in_10fold,Coverage
0,6.355100e-02,-0.0970,0.0000,0.00
1,1.089415e+06,0.6686,0.0871,0.56
2,2.178829e+06,0.6781,0.0686,0.74
3,3.268244e+06,0.6589,0.0600,0.86
4,4.357658e+06,0.6501,0.0618,0.92
5,5.447072e+06,0.6426,0.0550,0.95
6,6.536487e+06,0.6377,0.0468,0.97
7,7.625901e+06,0.6362,0.0478,0.98
8,8.715316e+06,0.6354,0.0478,0.99
9,9.804730e+06,0.6349,0.0481,0.99


Leverage-based AD 最佳阈值结果：
threshold            2.178829e+06
mean_R2_in_10fold    6.781000e-01
std_R2_in_10fold     6.860000e-02
Coverage             7.400000e-01
Name: 2, dtype: float64


In [2]:
results_df.to_csv("./Leverage-based.csv")